# Assistente de Contratos com RAG
### Versão corrigida — modelos gratuitos (HuggingFace)

In [ ]:
# ============================================================
# BLOCO 1: INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q sentence-transformers faiss-cpu pypdf
!pip install -q transformers accelerate huggingface_hub
print('Instalação concluída')


In [ ]:
# ============================================================
# BLOCO 2: IMPORTS
# ============================================================
import os
import random
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

print('Imports concluídos')


In [ ]:
# ============================================================
# BLOCO 3: LOGIN HUGGINGFACE (opcional)
# ============================================================
# Necessário apenas para modelos privados. TinyLlama é público.
# from huggingface_hub import login
# login('hf_SEU_TOKEN_AQUI')  # Nunca versione seu token
print('Bloco de login — descomente se necessário')


In [ ]:
# ============================================================
# BLOCO 4: CARREGAMENTO DOS PDFs
# ============================================================
arquivos = [
    'contrato_aluguel_1.pdf',
    'contrato_aluguel_2.pdf',
    'contrato_servico_1.pdf'
]

documentos = []
for arquivo in arquivos:
    if os.path.exists(arquivo):
        loader = PyPDFLoader(arquivo)
        docs = loader.load()
        for doc in docs:
            doc.metadata['tipo_contrato'] = 'aluguel' if 'aluguel' in arquivo else 'servico'
        documentos.extend(docs)
        print(f'📖 {arquivo} carregado ({len(docs)} páginas)')
    else:
        print(f'⚠️  {arquivo} não encontrado — pulando')

print(f'\nTotal de páginas carregadas: {len(documentos)}')


In [ ]:
# ============================================================
# BLOCO 5: DIVISÃO EM CHUNKS E CATEGORIZAÇÃO
# ============================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(documentos)
print(f'Total de chunks criados: {len(chunks)}')

# Categorização semântica por palavras-chave
for chunk in chunks:
    texto = chunk.page_content.lower()
    if any(k in texto for k in ['pagamento', 'valor', 'preço', 'aluguel']):
        chunk.metadata['categoria'] = 'pagamento'
    elif any(k in texto for k in ['rescisão', 'cancelamento', 'distrato']):
        chunk.metadata['categoria'] = 'rescisao'
    elif any(k in texto for k in ['multa', 'penalidade', 'juros']):
        chunk.metadata['categoria'] = 'multa'
    elif any(k in texto for k in ['prazo', 'vigência', 'validade']):
        chunk.metadata['categoria'] = 'prazo'
    elif any(k in texto for k in ['obrigação', 'responsabilidade', 'deveres']):
        chunk.metadata['categoria'] = 'obrigacoes'
    elif 'confidencialidade' in texto:
        chunk.metadata['categoria'] = 'confidencialidade'
    else:
        chunk.metadata['categoria'] = 'geral'

print('Categorização concluída')

# Visualizar 2 chunks aleatórios para validação
for i, chunk in enumerate(random.sample(chunks, min(2, len(chunks))), 1):
    print(f'\n--- Chunk {i} ---')
    print(f'Metadados: {chunk.metadata}')
    print(f'Conteúdo: {chunk.page_content[:200]}...')


In [ ]:
# ============================================================
# BLOCO 6: EMBEDDINGS E VECTORSTORE (FAISS)
# ============================================================
# all-MiniLM-L6-v2: modelo leve (~80MB), rápido e gratuito
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},       # troque por 'cuda' se tiver GPU
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

print('Vectorstore FAISS criado com sucesso!')
print(f'Total de vetores indexados: {vectorstore.index.ntotal}')


In [ ]:
# ============================================================
# BLOCO 7: LLM GRATUITO — TinyLlama (CPU-friendly)
# ============================================================
# max_new_tokens=80: suficiente para 2 frases objetivas.
# NÃO definir max_length junto — causa aviso de conflito.
# return_full_text=False: retorna só a resposta gerada.

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32   # float32 para CPU; use float16 se tiver GPU
)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    max_length=None,          # ⚡ sem max_length — elimina o aviso de conflito
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False      # retorna só o texto gerado, sem repetir o prompt
)

llm = HuggingFacePipeline(pipeline=pipe)
print('LLM carregado com sucesso!')


In [ ]:
# ============================================================
# BLOCO 8: PROMPT TEMPLATE
# ============================================================
# Regras do prompt:
# 1. 'Resposta:' como marcador — limpa vazamento do prompt na limpeza
# 2. Maximo 2 frases — evita alucinacao e lixo no final
# 3. Contexto antes da pergunta — modelo foca melhor no contexto

prompt_template = """Responda com base no contexto abaixo.
Se não encontrar a resposta, diga: Não encontrei essa informação no contrato.

Responda em no máximo 2 frases.

Contexto:
{context}

Pergunta: {question}

Resposta:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=['context', 'question']
)

print('Prompt configurado')


In [ ]:
# ============================================================
# BLOCO 9: FUNÇÕES DO AGENTE RAG
# ============================================================

def criar_retriever(categoria=None, k=3):
    """Cria retriever com ou sem filtro de categoria."""
    search_kwargs = {'k': k}
    if categoria:
        search_kwargs['filter'] = {'categoria': categoria}
    return vectorstore.as_retriever(search_kwargs=search_kwargs)


def limpar_resposta(texto):
    if 'Resposta:' in texto:
        texto = texto.split('Resposta:')[-1]
    return texto.strip()


def classificar_risco(resposta):
    resposta_lower = resposta.lower()
    
    if "multa" in resposta_lower and "proporcional" in resposta_lower:
        risco = "🟡 Risco moderado (multa proporcional)"
    elif "multa alta" in resposta_lower or "%" in resposta_lower:
        risco = "🔴 Alto risco financeiro"
    elif "rescis" in resposta_lower:
        risco = "🟡 Atenção à rescisão"
    else:
        risco = "🟢 Baixo risco"
        
    return risco
    elif 'multa alta' in t or '%' in t or 'multa' in t and any(v in t for v in ['50', '100', '3x', '10x']):
        return '🔴 Alto risco financeiro'
    elif 'rescis' in t or 'cancelamento' in t:
        return '🟡 Atenção à rescisão'
    elif 'obrigação' in t or 'responsabilidade' in t:
        return '🟡 Atenção às obrigações'
    else:
        return '🟢 Baixo risco'


def extrair_clausulas_simples(texto):
    """Identifica cláusulas-chave mencionadas na resposta."""
    t = texto.lower()
    clausulas = {
        'multa':    'Não identificado',
        'prazo':    'Não identificado',
        'rescisao': 'Não identificado'
    }
    if 'multa' in t or 'penalidade' in t:
        clausulas['multa'] = 'Possui cláusula de multa'
    if 'prazo' in t or 'vigência' in t or 'vigencia' in t:
        clausulas['prazo'] = 'Possui definição de prazo'
    if 'rescis' in t or 'cancelamento' in t:
        clausulas['rescisao'] = 'Possui cláusula de rescisão'
    return clausulas


def perguntar_contrato(pergunta, categoria=None):
    """Função principal do assistente RAG de contratos."""
    retriever = criar_retriever(categoria)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type='stuff',
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={'prompt': PROMPT}
    )

    resposta = qa_chain.invoke({'query': pergunta})
    # ======= PRINT PARA DEBUG DO CONTEXTO =======
    fontes = resposta['source_documents']
    contexto = "\n".join([d.page_content for d in fontes])
    print("\n" + "="*50)
    print("📋 DEBUG - Contexto recuperado:")
    print(contexto)
    print("="*50 + "\n")
    # ============================================

    texto_limpo = limpar_resposta(resposta['result'])
    risco = classificar_risco(texto_limpo)
    return texto_limpo, risco, resposta['source_documents']


print('Funções definidas e prontas para uso')


In [ ]:
# ============================================================
# BLOCO 10: TESTES DO ASSISTENTE DE CONTRATOS
# ============================================================

def verificar_confianca(resposta, fontes):
    """
    Alerta quando a resposta pode ser generalização do modelo
    e não está ancorada no texto real dos contratos.
    Compara se palavras-chave da resposta aparecem nas fontes.
    """
    palavras_resposta = set(resposta.lower().split())
    texto_fontes = ' '.join(f.page_content.lower() for f in fontes)
    palavras_fontes = set(texto_fontes.split())
    # Palavras com mais de 5 letras (ignora artigos e preposições)
    palavras_relevantes = {p for p in palavras_resposta if len(p) > 5}
    if not palavras_relevantes:
        return '✅ Sem palavras relevantes para verificar'
    sobreposicao = len(palavras_relevantes & palavras_fontes) / len(palavras_relevantes)
    if sobreposicao >= 0.5:
        return f'✅ Alta confiança — {sobreposicao:.0%} das palavras estão nos documentos'
    elif sobreposicao >= 0.25:
        return f'⚠️  Confiança moderada — {sobreposicao:.0%} das palavras encontradas (pode haver generalização)'
    else:
        return f'🚨 Baixa confiança — {sobreposicao:.0%} de sobreposição com os documentos (verifique manualmente)'


def executar_teste(pergunta, categoria=None):
    print('\n' + '='*60)
    print(f'🔎 PERGUNTA: {pergunta}')
    print(f'📂 CATEGORIA: {categoria if categoria else "Sem filtro"}')

    resposta, risco, fontes = perguntar_contrato(pergunta, categoria)

    print('\n📌 RESPOSTA:')
    print(resposta)

    print('\n⚠️  RISCO IDENTIFICADO:')
    print(risco)

    print('\n🔍 CONFIANÇA DA RESPOSTA:')
    print(verificar_confianca(resposta, fontes))

    clausulas = extrair_clausulas_simples(resposta)
    print('\n📊 CLÁUSULAS IDENTIFICADAS:')
    for chave, valor in clausulas.items():
        print(f'  - {chave.upper()}: {valor}')

    print(f'\n📄 FONTES CONSULTADAS: {len(fontes)}')
    for i, fonte in enumerate(fontes, 1):
        print(f'  {i}. {fonte.metadata.get("source", "desconhecida")} '
              f'| categoria: {fonte.metadata.get("categoria", "?")}')


# ── Executa os testes ──────────────────────────────────────────────
executar_teste('Qual é a multa por quebra de contrato?',   categoria='multa')
executar_teste('Existe penalidade por atraso?')
executar_teste('Qual é o prazo do contrato?',              categoria='prazo')
executar_teste('Como posso encerrar o contrato?',          categoria='rescisao')
executar_teste('Quais são as obrigações das partes?',      categoria='obrigacoes')
executar_teste('Explique os principais riscos do contrato')
